# PM2.5 Training & Inference (GPU P100)

**Prerequisites**: Run `preprocess_cpu.ipynb` first (CPU, no accelerator), save its output as a dataset, and attach it as input here.

Run this notebook with **GPU P100** accelerator.

In [1]:
import os
import gc
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ─── Paths ───
# IMPORTANT: Update this path to match your preprocessed dataset name on Kaggle
PREPROC_DIR = '/kaggle/input/datasets/jainilbavishi/pm25-preprocessed'  # <-- Change if your dataset name differs
OUT_DIR     = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

# ─── Constants ───
MONTHS = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']
ALL_FEATURES = ['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain',
                'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio']
N_FEATURES = len(ALL_FEATURES)
T_IN       = 10
T_OUT      = 16
WINDOW     = T_IN + T_OUT  # 26

# ─── Hyperparameters ───
BATCH_SIZE     = 8
NUM_EPOCHS     = 80
LR             = 1e-3
WEIGHT_DECAY   = 1e-4
PATIENCE       = 15
EPISODE_WEIGHT = 5.0
SEED           = 42
VAL_FRAC       = 0.2
STRIDE         = 1

# FNO architecture
FNO_WIDTH   = 64
FNO_MODES   = 16
FNO_BLOCKS  = 4
TEMP_HIDDEN = 64

# ─── Reproducibility ───
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ─── Verify preprocessed data ───
required = ['norm_stats.npy', 'test_data.npy'] + \
           [f'{m}_data.npy' for m in MONTHS] + \
           [f'{m}_episode_mask.npy' for m in MONTHS]
for f in required:
    path = os.path.join(PREPROC_DIR, f)
    assert os.path.exists(path), f'Missing: {path}. Check PREPROC_DIR and that dataset is attached.'
print('All preprocessed files found.')

# ─── Load norm stats ───
norm_stats = np.load(os.path.join(PREPROC_DIR, 'norm_stats.npy'), allow_pickle=True).item()

def denormalize_cpm25(arr):
    s = norm_stats['cpm25']
    return arr * (s['p99'] - s['p1']) + s['p1']

print(f'cpm25 denorm range: [{norm_stats["cpm25"]["p1"]:.2f}, {norm_stats["cpm25"]["p99"]:.2f}]')

Device: cuda
All preprocessed files found.
cpm25 denorm range: [0.72, 257.30]


## 1. Dataset & DataLoader

In [2]:
class PM25Dataset(Dataset):
    """Loads normalized monthly arrays via mmap and creates windows on-the-fly."""
    def __init__(self, month_data_list, month_mask_list, sample_indices):
        """
        month_data_list: list of mmap arrays, each (T, 140, 124, 16) float16
        month_mask_list: list of mmap arrays, each (T, 140, 124) uint8
        sample_indices:  list of (month_idx, time_offset) tuples
        """
        self.data = month_data_list
        self.masks = month_mask_list
        self.indices = sample_indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        mi, t = self.indices[idx]

        # Slice window on-the-fly from monthly array
        window = self.data[mi][t:t+WINDOW].astype(np.float32)  # (26, 140, 124, 16)
        mask = self.masks[mi][t+T_IN:t+WINDOW].astype(np.float32)  # (16, 140, 124)

        x = torch.from_numpy(window[:T_IN].copy())            # (10, 140, 124, 16)
        y = torch.from_numpy(window[T_IN:, :, :, 0].copy())   # (16, 140, 124) - cpm25 only
        m = torch.from_numpy(mask.copy())                      # (16, 140, 124)

        y = y.permute(1, 2, 0)  # (140, 124, 16)
        m = m.permute(1, 2, 0)  # (140, 124, 16)
        return x, y, m


# ─── Load monthly arrays via memory-map (no RAM cost) ───
month_data_list = []
month_mask_list = []
for month in MONTHS:
    d = np.load(os.path.join(PREPROC_DIR, f'{month}_data.npy'), mmap_mode='r')
    m = np.load(os.path.join(PREPROC_DIR, f'{month}_episode_mask.npy'), mmap_mode='r')
    month_data_list.append(d)
    month_mask_list.append(m)
    print(f'{month}: data={d.shape}, mask={m.shape}')

# ─── Build sample index: (month_idx, time_offset) for all valid windows ───
all_indices = []
for mi, month in enumerate(MONTHS):
    T_month = month_data_list[mi].shape[0]
    for t in range(0, T_month - WINDOW + 1, STRIDE):
        all_indices.append((mi, t))

N_total = len(all_indices)
print(f'Total samples: {N_total}')

# ─── Shuffle and split ───
perm = np.random.permutation(N_total)
n_val = int(N_total * VAL_FRAC)
val_indices = [all_indices[i] for i in perm[:n_val]]
train_indices = [all_indices[i] for i in perm[n_val:]]

train_ds = PM25Dataset(month_data_list, month_mask_list, train_indices)
val_ds   = PM25Dataset(month_data_list, month_mask_list, val_indices)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)

print(f'Train: {len(train_ds)} samples ({len(train_loader)} batches)')
print(f'Val:   {len(val_ds)} samples ({len(val_loader)} batches)')

APRIL_16: data=(715, 140, 124, 16), mask=(715, 140, 124)
JULY_16: data=(739, 140, 124, 16), mask=(739, 140, 124)
OCT_16: data=(739, 140, 124, 16), mask=(739, 140, 124)
DEC_16: data=(739, 140, 124, 16), mask=(739, 140, 124)
Total samples: 2832
Train: 2266 samples (284 batches)
Val:   566 samples (71 batches)


## 2. Model Architecture - Improved FNO2D

In [3]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        scale = 1 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))

    def compl_mul2d(self, inp, weights):
        return torch.einsum('bixy,ioxy->boxy', inp, weights)

    def forward(self, x):
        B = x.shape[0]
        # cuFFT doesn't support half precision for non-power-of-2 sizes (140x124)
        # Force float32 for FFT operations
        with torch.amp.autocast('cuda', enabled=False):
            x = x.float()
            x_ft = torch.fft.rfft2(x)

            out_ft = torch.zeros(B, self.out_channels, x.size(-2), x.size(-1)//2 + 1,
                                dtype=torch.cfloat, device=x.device)
            out_ft[:, :, :self.modes1, :self.modes2] = \
                self.compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
            out_ft[:, :, -self.modes1:, :self.modes2] = \
                self.compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)

            return torch.fft.irfft2(out_ft, s=(x.size(-2), x.size(-1)))


class ChannelMLP(nn.Module):
    def __init__(self, in_ch, hidden_ch, out_ch):
        super().__init__()
        self.fc1 = nn.Conv2d(in_ch, hidden_ch, 1)
        self.fc2 = nn.Conv2d(hidden_ch, out_ch, 1)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class FNOBlock(nn.Module):
    def __init__(self, width, modes):
        super().__init__()
        self.spectral = SpectralConv2d(width, width, modes, modes)
        self.pointwise = nn.Conv2d(width, width, 1)
        self.norm = nn.InstanceNorm2d(width)
        self.mlp = ChannelMLP(width, width // 2, width)

    def forward(self, x):
        residual = x
        x = self.spectral(x) + self.pointwise(x)
        x = self.norm(x)
        x = F.gelu(x)
        x = x + self.mlp(x)
        x = x + residual
        return x


class TemporalMixer(nn.Module):
    def __init__(self, n_features, hidden_dim):
        super().__init__()
        self.conv1 = nn.Conv3d(n_features, hidden_dim, kernel_size=(3, 1, 1), padding=(1, 0, 0))
        self.conv2 = nn.Conv3d(hidden_dim, hidden_dim, kernel_size=(3, 1, 1), padding=(1, 0, 0))
        self.pool = nn.AdaptiveAvgPool3d((1, None, None))

    def forward(self, x):
        x = x.permute(0, 4, 1, 2, 3)  # (B, F, T, H, W)
        x = F.gelu(self.conv1(x))
        x = F.gelu(self.conv2(x))
        x = self.pool(x).squeeze(2)
        return x


class ImprovedFNO2D(nn.Module):
    def __init__(self, n_features=N_FEATURES, t_in=T_IN, t_out=T_OUT,
                 width=FNO_WIDTH, modes=FNO_MODES, n_blocks=FNO_BLOCKS, temp_hidden=TEMP_HIDDEN):
        super().__init__()
        self.temporal_mixer = TemporalMixer(n_features, temp_hidden)
        in_ch = temp_hidden + 2
        self.lifting = ChannelMLP(in_ch, width * 2, width)
        self.blocks = nn.ModuleList([FNOBlock(width, modes) for _ in range(n_blocks)])
        self.proj1 = nn.Conv2d(width, 128, 1)
        self.proj2 = nn.Conv2d(128, t_out, 1)

    def get_grid(self, B, H, W, device):
        gridx = torch.linspace(0, 1, H, device=device).view(1, 1, H, 1).expand(B, 1, H, W)
        gridy = torch.linspace(0, 1, W, device=device).view(1, 1, 1, W).expand(B, 1, H, W)
        return torch.cat([gridx, gridy], dim=1)

    def forward(self, x):
        B, T, H, W, C = x.shape  # C not F, avoid shadowing torch.nn.functional
        x = self.temporal_mixer(x)
        grid = self.get_grid(B, H, W, x.device)
        x = torch.cat([x, grid], dim=1)
        x = self.lifting(x)
        for block in self.blocks:
            x = block(x)
        x = F.gelu(self.proj1(x))
        x = self.proj2(x)
        return x.permute(0, 2, 3, 1)  # (B, H, W, T_OUT)


model = ImprovedFNO2D().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')

Model parameters: 8,464,720


## 3. Episode-Aware Loss Function

In [4]:
class CompositeLoss(nn.Module):
    def __init__(self, episode_weight=EPISODE_WEIGHT, smape_w=0.5, l2_w=0.5):
        super().__init__()
        self.episode_weight = episode_weight
        self.smape_w = smape_w
        self.l2_w = l2_w

    def forward(self, pred, target, episode_mask):
        weights = 1.0 + episode_mask * (self.episode_weight - 1.0)
        mse = (weights * (pred - target) ** 2).mean()
        denom = torch.abs(pred) + torch.abs(target) + 1e-8
        smape = (weights * torch.abs(pred - target) / denom).mean()
        return self.l2_w * mse + self.smape_w * smape

criterion = CompositeLoss()
print(f'Loss: CompositeLoss (episode_weight={EPISODE_WEIGHT})')

Loss: CompositeLoss (episode_weight=5.0)


## 4. Training Loop

In [5]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

best_val_loss = float('inf')
patience_counter = 0
best_ckpt_path = os.path.join(OUT_DIR, 'best_model.pt')
train_log = []

# Print GPU info
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Training: {len(train_ds)} samples, {len(train_loader)} batches/epoch')
print(f'Validation: {len(val_ds)} samples, {len(val_loader)} batches/epoch')
print(f'Model: {sum(p.numel() for p in model.parameters()):,} params')
print(f'Hyperparams: lr={LR}, batch={BATCH_SIZE}, epochs={NUM_EPOCHS}, patience={PATIENCE}')
print(f'Episode weight: {EPISODE_WEIGHT}')
print('='*80)

total_train_time = 0

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # ── Train ──
    model.train()
    train_loss_sum = 0.0
    train_count = 0

    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch:2d}/{NUM_EPOCHS} [Train]',
                      leave=False, ncols=100)
    for x, y, m in train_pbar:
        x, y, m = x.to(DEVICE), y.to(DEVICE), m.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        pred = model(x)
        loss = criterion(pred, y, m)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss_sum += loss.item() * x.size(0)
        train_count += x.size(0)
        train_pbar.set_postfix(loss=f'{loss.item():.4f}')

    scheduler.step()
    train_loss = train_loss_sum / train_count

    # ── Validate ──
    model.eval()
    val_loss_sum = 0.0
    val_count = 0

    val_pbar = tqdm(val_loader, desc=f'Epoch {epoch:2d}/{NUM_EPOCHS} [Val]  ',
                    leave=False, ncols=100)
    with torch.no_grad():
        for x, y, m in val_pbar:
            x, y, m = x.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
            pred = model(x)
            loss = criterion(pred, y, m)
            val_loss_sum += loss.item() * x.size(0)
            val_count += x.size(0)
            val_pbar.set_postfix(loss=f'{loss.item():.4f}')

    val_loss = val_loss_sum / val_count
    elapsed = time.time() - t0
    total_train_time += elapsed
    lr_now = optimizer.param_groups[0]['lr']

    # GPU memory stats
    gpu_mem = torch.cuda.max_memory_allocated() / 1e9

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_loss': val_loss}, best_ckpt_path)
    else:
        patience_counter += 1

    marker = ' *BEST*' if improved else ''
    print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
          f'Train: {train_loss:.6f} | Val: {val_loss:.6f}{marker} | '
          f'LR: {lr_now:.2e} | {elapsed:.0f}s | '
          f'GPU: {gpu_mem:.1f}GB | '
          f'Patience: {patience_counter}/{PATIENCE}')

    train_log.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                      'lr': lr_now, 'time': elapsed})

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print('='*80)
print(f'Best val loss: {best_val_loss:.6f}')
print(f'Total training time: {total_train_time/60:.1f} minutes ({total_train_time/3600:.1f} hours)')
print(f'Avg time/epoch: {total_train_time/epoch:.0f}s')

with open(os.path.join(OUT_DIR, 'train_log.json'), 'w') as f:
    json.dump(train_log, f, indent=2)

GPU: Tesla T4
GPU Memory: 15.6 GB
Training: 2266 samples, 284 batches/epoch
Validation: 566 samples, 71 batches/epoch
Model: 8,464,720 params
Hyperparams: lr=0.001, batch=8, epochs=80, patience=15
Episode weight: 5.0


Epoch   1/80 | Train: 0.161335 | Val: 0.122251 *BEST* | LR: 9.94e-04 | 127s | GPU: 2.9GB | Patience: 0/15


Epoch   2/80 | Train: 0.112058 | Val: 0.099807 *BEST* | LR: 9.76e-04 | 127s | GPU: 2.9GB | Patience: 0/15


Epoch   3/80 | Train: 0.097015 | Val: 0.088999 *BEST* | LR: 9.46e-04 | 126s | GPU: 2.9GB | Patience: 0/15


Epoch   4/80 | Train: 0.087275 | Val: 0.087710 *BEST* | LR: 9.05e-04 | 126s | GPU: 2.9GB | Patience: 0/15


...

Epoch  31/80 | Train: 0.056768 | Val: 0.053673 | LR: 8.25e-04 | 126s | GPU: 2.9GB | Patience: 11/15


Epoch  32/80 | Train: 0.052518 | Val: 0.050984 | LR: 7.94e-04 | 127s | GPU: 2.9GB | Patience: 12/15


Epoch  33/80 | Train: 0.052456 | Val: 0.055386 | LR: 7.61e-04 | 127s | GPU: 2.9GB | Patience: 13/15


Epoch  34/80 | Train: 0.052841 | Val: 0.055434 | LR: 7.27e-04 | 127s | GPU: 2.9GB | Patience: 14/15


Epoch  35/80 | Train: 0.050626 | Val: 0.051734 | LR: 6.91e-04 | 126s | GPU: 2.9GB | Patience: 15/15

Early stopping at epoch 35.
Best val loss: 0.047694
Total training time: 73.8 minutes (1.2 hours)
Avg time/epoch: 126s


## 5. Inference on Test Data

In [6]:
# Load best model
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best model from epoch {ckpt['epoch']} (val_loss={ckpt['val_loss']:.6f})")

# Load preprocessed test data (already normalized, float16)
test_data = np.load(os.path.join(PREPROC_DIR, 'test_data.npy'))  # (N_test, 10, 140, 124, 16)
N_test = test_data.shape[0]
print(f'Test samples: {N_test}, shape: {test_data.shape}')

# Batch inference
predictions = []
with torch.no_grad():
    for i in tqdm(range(0, N_test, BATCH_SIZE), desc='Inference'):
        batch = torch.from_numpy(test_data[i:i+BATCH_SIZE].astype(np.float32)).to(DEVICE)
        pred = model(batch)
        predictions.append(pred.cpu().numpy())

predictions = np.concatenate(predictions, axis=0)
print(f'Raw predictions shape: {predictions.shape}')

# Denormalize and clip
predictions = denormalize_cpm25(predictions)
predictions = np.clip(predictions, 0, None)

# Save
preds_path = os.path.join(OUT_DIR, 'preds.npy')
np.save(preds_path, predictions.astype(np.float32))
print(f'Saved: {preds_path}')
print(f'Shape: {predictions.shape}')
print(f'Range: [{predictions.min():.2f}, {predictions.max():.2f}], Mean: {predictions.mean():.2f}')

Loaded best model from epoch 20 (val_loss=0.047694)
Test samples: 218, shape: (218, 10, 140, 124, 16)


Inference: 100%|██████████| 28/28 [00:06<00:00,  4.45it/s]


Raw predictions shape: (218, 140, 124, 16)
Saved: /kaggle/working/preds.npy
Shape: (218, 140, 124, 16)
Range: [0.00, 371.84], Mean: 35.09


In [7]:
# Final verification
preds = np.load(preds_path)
print(f'preds.npy shape: {preds.shape}')
assert preds.shape == (218, 140, 124, 16), f'Wrong shape: {preds.shape}, expected (218, 140, 124, 16)'
assert preds.dtype == np.float32, f'Wrong dtype: {preds.dtype}, expected float32'
assert preds.min() >= 0, f'Negative values: min={preds.min()}'
assert not np.any(np.isnan(preds)), 'NaN values found in predictions!'
print(f'Value range: [{preds.min():.2f}, {preds.max():.2f}]')
print('All checks passed!')

preds.npy shape: (218, 140, 124, 16)
Value range: [0.00, 371.84]
All checks passed!
